# Securing BGP Origins with RPKI

In this notebook you will check who is allowed to originate a prefix the same way a
validating router does, then measure how much of the real internet that check covers:

- implement **Route Origin Validation (ROV)** and classify real announcements against a real
  snapshot of validated RPKI data
- scale it to **every announced IPv4 prefix** on one day, about 1.1 million (prefix, origin) pairs
- measure ROA coverage per network, and compare **MANRS participants** with everyone else
- count how many networks have taken the first deployable step of **BGPsec**, next to **ASPA**

Work through the tasks in order alongside [Tasks.md](Tasks.md). Replace each `# YOUR CODE HERE`
section with your code, and answer each question in the slot directly below it. The datasets are
described in [Datasets.md](Datasets.md).

---
## Task 0: Setup and Data Access

Run every cell in this section, in order. The first run downloads about 350 MB of VRPs, the day's
routing table, the MANRS participant list, and a ~100 MB live RPKI snapshot, caching what it keeps
under `data/`; later runs read the cached copies and take seconds.

In [ ]:
%pip install pandas requests matplotlib

In [ ]:
import gzip
import io
import ipaddress
import json
import lzma
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

# The five RPKI trust anchors, one per RIR.
TRUST_ANCHORS = ["afrinic", "ripencc", "arin", "lacnic", "apnic"]

# Task 2 classifies a handful of real announcements against this day's VRPs.
ROV_DATE = "2023-03-01"

# Tasks 3-6 compare this day's routing table with this day's VRPs. Both archives
# publish daily; any recent day works.
SNAPSHOT_DATE = "2026-07-01"

VRP_ARCHIVE_URL = "https://ftp.ripe.net/ripe/rpki/{ta}.tal/{yyyy}/{mm}/{dd}/roas.csv.xz"
PFX2AS_URL = (
    "https://publicdata.caida.org/datasets/routing/routeviews-prefix2as/"
    "{yyyy}/{mm}/routeviews-rv2-{yyyy}{mm}{dd}-1200.pfx2as.gz"
)
MANRS_ASNS_URL = "https://observatory.manrs.org/api/v2/asns"
RPKI_CLIENT_URL = "https://console.rpki-client.org/rpki.json"

# Downloaded files are cached here so re-running the notebook doesn't re-fetch them.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Task 2 snapshot:     {ROV_DATE}")
print(f"Tasks 3-6 snapshot:  {SNAPSHOT_DATE}")

### The VRP loader (provided)

`load_rpki_snapshot(date)` downloads one `roas.csv.xz` per trust anchor for the given day,
normalizes the RIPE column names and value formats (stripping the `AS` prefix off ASNs and filling
a blank max length with the prefix's own length), and returns one row per VRP with the columns
`prefix`, `max_length`, `asn`, `trust_anchor`, `family`, and `date`.

In [ ]:
def _fetch_vrp_csv(trust_anchor: str, date: str) -> bytes:
    """Download (or read from cache) one trust anchor's VRP file for one day."""
    yyyy, mm, dd = date.split("-")
    cache_path = DATA_DIR / f"{trust_anchor}-{date}.csv"

    if cache_path.exists():
        return cache_path.read_bytes()

    url = VRP_ARCHIVE_URL.format(ta=trust_anchor, yyyy=yyyy, mm=mm, dd=dd)
    response = requests.get(url, timeout=180)
    response.raise_for_status()

    raw = lzma.decompress(response.content)
    cache_path.write_bytes(raw)
    return raw


def _normalize(df: pd.DataFrame, trust_anchor: str, date: str) -> pd.DataFrame:
    """Map RIPE's CSV columns onto the schema described above."""
    # RIPE's header has varied over the years; match it case-insensitively.
    lookup = {c.strip().lower(): c for c in df.columns}
    asn_col = lookup.get("asn")
    prefix_col = lookup.get("ip prefix", lookup.get("prefix"))
    maxlen_col = lookup.get("max length", lookup.get("maxlength"))

    out = pd.DataFrame()
    out["prefix"] = df[prefix_col].astype(str).str.strip()

    # ASNs appear as "AS7377" or "7377" depending on the vintage of the file.
    out["asn"] = (
        df[asn_col]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"^AS", "", regex=True)
        .astype(int)
    )

    # A blank max length means "exactly the prefix's own length".
    prefix_len = out["prefix"].str.split("/").str[1].astype(int)
    out["max_length"] = (
        pd.to_numeric(df[maxlen_col], errors="coerce").fillna(prefix_len).astype(int)
    )

    out["trust_anchor"] = trust_anchor
    out["family"] = out["prefix"].str.contains(":").map({True: 6, False: 4})
    out["date"] = date

    return out[["prefix", "max_length", "asn", "trust_anchor", "family", "date"]]


def load_rpki_snapshot(date: str = SNAPSHOT_DATE, trust_anchors=TRUST_ANCHORS) -> pd.DataFrame:
    """Return one row per VRP across all trust anchors for the given day."""
    frames = []
    for ta in trust_anchors:
        raw = _fetch_vrp_csv(ta, date)
        df = pd.read_csv(io.BytesIO(raw))
        frames.append(_normalize(df, ta, date))
        print(f"  {ta:<8} {len(frames[-1]):>9,} VRPs")

    return pd.concat(frames, ignore_index=True)

In [ ]:
print(f"Loading VRPs for {ROV_DATE} ...")
rov_vrps = load_rpki_snapshot(ROV_DATE)

print(f"\nTotal: {len(rov_vrps):,} VRPs")
print("\nVRPs per address family:")
print(rov_vrps["family"].value_counts().to_string())
rov_vrps.head()

In [ ]:
print(f"Loading VRPs for {SNAPSHOT_DATE} ...")
vrps = load_rpki_snapshot(SNAPSHOT_DATE)

# Sanity check: by mid-2026 the five trust anchors together publish close to a
# million VRPs, with ripencc, apnic, and arin the three largest.
print(f"\nTotal: {len(vrps):,} VRPs")
vrps.head()

### The routing-table loader (provided)

`load_prefix2as_snapshot(date)` reads CAIDA's Routeviews prefix-to-AS file for the day and returns
one row per announced **(prefix, origin)** pair. A prefix observed with several origins becomes
several rows, because ROV classifies each pair separately.

In [ ]:
def load_prefix2as_snapshot(date: str = SNAPSHOT_DATE) -> pd.DataFrame:
    """Return one row per announced (prefix, origin) pair observed on the given day."""
    yyyy, mm, dd = date.split("-")
    cache_path = DATA_DIR / f"pfx2as-{date}.csv.gz"

    if cache_path.exists():
        return pd.read_csv(cache_path)

    url = PFX2AS_URL.format(yyyy=yyyy, mm=mm, dd=dd)
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    text = gzip.decompress(response.content).decode("utf-8", errors="ignore")

    rows = []
    for line in text.splitlines():
        fields = line.split()
        if len(fields) < 3:
            continue
        prefix = f"{fields[0]}/{fields[1]}"
        # Multi-origin rows list several ASes ("64500_64501"); one row per origin.
        for token in re.split(r"[_,{}]+", fields[2]):
            if token.isdigit():
                rows.append((prefix, int(token)))

    bgp = pd.DataFrame(rows, columns=["prefix", "origin"]).drop_duplicates(ignore_index=True)
    bgp["date"] = date
    bgp.to_csv(cache_path, index=False, compression="gzip")
    return bgp

In [ ]:
bgp = load_prefix2as_snapshot()

# Sanity check: by mid-2026 you should see roughly 1.1 million pairs.
print(f"{len(bgp):,} announced (prefix, origin) pairs on {SNAPSHOT_DATE}")
bgp.head()

### The MANRS participant list (provided)

`load_manrs_asns()` returns the ASNs claimed by approved MANRS participants.

> **Note:** the endpoint returns the *current* membership, not membership as of `SNAPSHOT_DATE`.
> The list is cached under `data/`, so re-runs keep using the list from the day you first ran
> this cell.

In [ ]:
def load_manrs_asns() -> set:
    """Return the set of ASNs claimed by approved MANRS participants (fetched live)."""
    cache_path = DATA_DIR / "manrs_asns.json"

    if cache_path.exists():
        return set(json.loads(cache_path.read_text()))

    response = requests.get(MANRS_ASNS_URL, timeout=60)
    response.raise_for_status()
    asns = response.json()["asns"]
    cache_path.write_text(json.dumps(asns))
    return set(asns)

In [ ]:
manrs_asns = load_manrs_asns()
print(f"{len(manrs_asns):,} ASNs are claimed by approved MANRS participants")

### The live validated-RPKI summary (provided)

`load_bgpsec_snapshot()` fetches rpki-client's full validated output (~100 MB, validated minutes
ago rather than on `SNAPSHOT_DATE`) and keeps only the ROA, VRP, and ASPA counts and the BGPsec
router keys. It caches that small summary under `data/`, so it downloads once.

In [ ]:
def load_bgpsec_snapshot(force_refresh: bool = False) -> dict:
    """Fetch the live validated-RPKI summary and keep only the small parts we need."""
    cache_path = DATA_DIR / "bgpsec_snapshot.json"

    if cache_path.exists() and not force_refresh:
        return json.loads(cache_path.read_text())

    print("Downloading the full validated-RPKI snapshot (~100 MB, one time) ...")
    response = requests.get(RPKI_CLIENT_URL, timeout=600)
    response.raise_for_status()
    data = response.json()

    snapshot = {
        "buildtime": data["metadata"]["buildtime"],
        "roas": data["metadata"]["roas"],
        "vrps": data["metadata"]["vrps"],
        "aspas": data["metadata"]["aspas"],
        "bgpsec_keys": data["bgpsec_keys"],
    }
    cache_path.write_text(json.dumps(snapshot, indent=1))
    return snapshot

In [ ]:
snapshot = load_bgpsec_snapshot()
print(f"Validated RPKI as of {snapshot['buildtime']}: {snapshot['roas']:,} ROAs, "
      f"{len(snapshot['bgpsec_keys']):,} BGPsec router keys")

---
## Task 1: RPKI Concepts

Detailed guidance: [Task 1 — Concepts](Task-1-concepts.md)

These questions check the RPKI background in [Introduction.md](Introduction.md). There is no code in this task.

### Task 1.1 — ROAs and ROV

**Q1** In one sentence each, what is a **ROA** and what is **ROV**?

*Your answer for Q1:*

Replace this line with your answer.

**Q2** A router runs ROV on a route and gets **Not Found**. Why is dropping every Not Found route a bad idea today?

*Your answer for Q2:*

Replace this line with your answer.

### Task 1.2 — Max Length

**Q3** An operator announces `203.0.113.0/24` and also, sometimes, `203.0.113.0/26`. What should the **Max Length** on their ROA be, and what would break if they set it to `/24`?

*Your answer for Q3:*

Replace this line with your answer.

### Task 1.3 — What origin validation cannot see

**Q4** Why can a **prefix hijack** be stopped by RPKI, while a **forged-AS_PATH** attack that ends in the real origin can slip through as Valid?

*Your answer for Q4:*

Replace this line with your answer.

---
## Task 2: Route Origin Validation

Detailed guidance: [Task 2 — Route Origin Validation](Task-2-rov.md)

You will classify a handful of real announcements against the 2023-03-01 snapshot, `rov_vrps`. Each target is a real IP address, the prefix it was announced in, and the AS that originated that announcement: the two facts a router takes from the BGP update carrying the route.

### Task 2.1 — Implement `classify_rov`

In [ ]:
# YOUR CODE HERE
# Produce: classify_rov(announced_prefix: str, origin_asn: int, vrps: pd.DataFrame) -> str
#          returning exactly one of "Valid", "Invalid", or "Not Found".
#          vrps has the columns prefix, max_length, asn, trust_anchor, family, date.
# Hint: ipaddress.ip_network(a).subnet_of(ipaddress.ip_network(b)) is True when b covers a;
#       it raises TypeError across IPv4/IPv6, so compare .version (or the family column) first.
# Hint: iterate with vrps.itertuples(); one authorizing VRP is enough for "Valid", but you
#       must see every covering VRP before you may answer "Invalid".

Test your function on a tiny table you can reason about by hand before running it on 400,000 real VRPs. Every case below should pass; each one isolates a different part of the rule.

In [ ]:
TEST_VRPS = pd.DataFrame(
    [
        # prefix,           max_length, asn,   trust_anchor, family, date
        ("203.0.113.0/24", 24, 64500, "arin", 4, "test"),
        ("198.51.100.0/24", 26, 64501, "ripencc", 4, "test"),
        ("192.0.2.0/24", 24, 64502, "apnic", 4, "test"),
        ("192.0.2.0/24", 24, 64503, "lacnic", 4, "test"),  # same prefix, second origin
        ("2001:db8::/32", 48, 64504, "ripencc", 6, "test"),
    ],
    columns=["prefix", "max_length", "asn", "trust_anchor", "family", "date"],
)

TEST_CASES = [
    # (prefix, origin, expected, what it exercises)
    ("203.0.113.0/24", 64500, "Valid", "exact match on prefix and origin"),
    ("203.0.113.0/24", 66000, "Invalid", "covered, wrong origin"),
    ("203.0.113.128/25", 64500, "Invalid", "right origin, beyond max length"),
    ("203.0.113.128/25", 66000, "Invalid", "sub-prefix hijack"),
    ("198.51.100.192/26", 64501, "Valid", "more specific, but within max length"),
    ("198.51.100.192/28", 64501, "Invalid", "more specific than max length allows"),
    ("192.0.2.0/24", 64503, "Valid", "second covering VRP authorizes it"),
    ("2001:db8:1::/48", 64504, "Valid", "IPv6, within max length"),
    ("10.0.0.0/8", 64500, "Not Found", "no VRP covers this prefix at all"),
]

passed = 0
for prefix, origin, expected, description in TEST_CASES:
    actual = classify_rov(prefix, origin, TEST_VRPS)
    ok = actual == expected
    passed += ok
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {prefix:<18} AS{origin:<6} got {actual:<9} "
          f"expected {expected:<9} ({description})")

print(f"\n{passed}/{len(TEST_CASES)} cases passed")

### Task 2.2 — Classify real announcements

In [ ]:
# (ip_address, announced_prefix, origin_asn)
TARGETS = [
    ("8.8.8.8", "8.8.8.0/24", 15169),
    ("8.8.8.8", "8.8.8.0/24", 4134),
    ("1.1.1.1", "1.1.1.0/24", 13335),
    ("1.1.1.1", "1.1.1.0/24", 7007),
    ("192.0.66.20", "192.0.64.0/18", 2635),
    ("192.0.66.20", "192.0.66.0/24", 2635),
    ("192.0.66.200", "192.0.66.128/25", 2635),
    ("208.65.153.238", "208.65.152.0/22", 36040),
    ("208.65.153.238", "208.65.153.0/24", 17557),
    ("137.110.222.10", "137.110.0.0/16", 7377),
    ("198.51.100.7", "198.51.100.0/24", 64511),
]

print(f"{len(TARGETS)} announcements to classify")

In [ ]:
results = pd.DataFrame(
    [
        {
            "ip": ip,
            "announced_prefix": prefix,
            "origin_asn": origin,
            "rov_state": classify_rov(prefix, origin, rov_vrps),
        }
        for ip, prefix, origin in TARGETS
    ]
)

print(results["rov_state"].value_counts().to_string())
results

**Q5** Three of the targets announce the same address space from the same origin AS at three different prefix lengths (`/18`, `/24`, `/25`). Did all three get the same state? Explain the boundary in terms of the covering ROA's max length.

*Your answer for Q5:*

Replace this line with your answer.

**Q6** Two targets came back **Not Found**. What would a router configured with standard policy do with them, and why is accepting them still the safe choice today?

*Your answer for Q6:*

Replace this line with your answer.

### Task 2.3 — Diagnose an Invalid announcement

In [ ]:
# YOUR CODE HERE
# Produce: covering — a DataFrame holding the rows of rov_vrps whose prefix covers the
#          announced prefix of one announcement that came back "Invalid" in results.
# Hint: pick the prefix from results[results["rov_state"] == "Invalid"].
# Hint: reuse the same family check and subnet_of() test your classify_rov performs,
#       applied to rov_vrps["prefix"] to build a boolean mask.

**Q7** Pick an announcement that came back **Invalid**. Using the covering VRPs you listed, explain exactly which condition it failed — wrong origin, excessive length, or both.

*Your answer for Q7:*

Replace this line with your answer.

### Task 2.4 — Trust anchors

**Q8** Your `classify_rov` ignores the `trust_anchor` column entirely. Why is that correct? What would go wrong if you had filtered the VRPs by trust anchor first?

*Your answer for Q8:*

Replace this line with your answer.

---
## Task 3: ROV at Internet Scale

Detailed guidance: [Task 3 — Internet Scale](Task-3-internet-scale.md)

Classifying about 1.1 million announcements against about a million VRPs is far too much for a row scan per announcement, so the VRPs are indexed by exact prefix instead. The classification rule is unchanged from Task 2.

### Task 3.1 — Index the VRPs

In [ ]:
def build_vrp_index(vrps_df: pd.DataFrame):
    """Index IPv4 VRPs by (network address, prefix length) for fast covering lookups.

    Returns (table, lengths):
      table   maps (network_int, length) -> list of (vrp_prefix, asn, max_length)
      lengths is the sorted list of prefix lengths that occur in the table
    """
    v4 = vrps_df[vrps_df["family"] == 4]

    table = {}
    for prefix, asn, max_length in zip(v4["prefix"], v4["asn"], v4["max_length"]):
        network, _, length = prefix.partition("/")
        key = (int(ipaddress.IPv4Address(network)), int(length))
        table.setdefault(key, []).append((prefix, int(asn), int(max_length)))

    lengths = sorted({length for _, length in table})
    return table, lengths


def covering_vrps(prefix: str, vrp_index) -> list:
    """Return every VRP whose prefix covers `prefix` (IPv4 only).

    Each element is a tuple (vrp_prefix, asn, max_length).
    """
    table, lengths = vrp_index

    network, _, length = prefix.partition("/")
    address = int(ipaddress.IPv4Address(network))
    prefix_length = int(length)

    covering = []
    for vrp_length in lengths:
        if vrp_length > prefix_length:
            break  # a longer prefix can never cover a shorter one
        mask = (0xFFFFFFFF << (32 - vrp_length)) & 0xFFFFFFFF if vrp_length else 0
        hit = table.get((address & mask, vrp_length))
        if hit:
            covering.extend(hit)
    return covering

In [ ]:
vrp_index = build_vrp_index(vrps)

table, lengths = vrp_index
print(f"Indexed {len(table):,} distinct IPv4 VRP prefixes")
print(f"Prefix lengths present: /{lengths[0]} through /{lengths[-1]} "
      f"({len(lengths)} distinct lengths)")

### Task 3.2 — Implement `classify_rov_indexed`

In [ ]:
# YOUR CODE HERE
# Produce: classify_rov_indexed(prefix: str, origin_asn: int, vrp_index) -> str
#          returning exactly one of "Valid", "Invalid", or "Not Found".
#          vrp_index is the (table, lengths) pair returned by build_vrp_index().
# Hint: covering_vrps(prefix, vrp_index) returns a list of plain tuples
#       (vrp_prefix, asn, max_length) that already cover the prefix.
# Hint: an empty list means "Not Found"; otherwise check origin and max length only.

Before touching a million real announcements, make sure every case below passes.

In [ ]:
INDEX_TEST_VRPS = pd.DataFrame(
    [
        # prefix,           max_length, asn,   trust_anchor, family, date
        ("203.0.113.0/24", 24, 64500, "arin", 4, "test"),
        ("198.51.100.0/24", 26, 64501, "ripencc", 4, "test"),
        ("192.0.2.0/24", 24, 64502, "apnic", 4, "test"),
        ("192.0.2.0/24", 24, 64503, "lacnic", 4, "test"),  # same prefix, second origin
    ],
    columns=["prefix", "max_length", "asn", "trust_anchor", "family", "date"],
)
TEST_INDEX = build_vrp_index(INDEX_TEST_VRPS)

INDEX_TEST_CASES = [
    # (prefix, origin, expected, what it exercises)
    ("203.0.113.0/24", 64500, "Valid", "exact match on prefix and origin"),
    ("203.0.113.0/24", 66000, "Invalid", "covered, wrong origin"),
    ("203.0.113.128/25", 64500, "Invalid", "right origin, beyond max length"),
    ("198.51.100.192/26", 64501, "Valid", "more specific, but within max length"),
    ("198.51.100.192/28", 64501, "Invalid", "more specific than max length allows"),
    ("192.0.2.0/24", 64503, "Valid", "second covering VRP authorizes it"),
    ("10.0.0.0/8", 64500, "Not Found", "no VRP covers this prefix at all"),
]

passed = 0
for prefix, origin, expected, description in INDEX_TEST_CASES:
    actual = classify_rov_indexed(prefix, origin, TEST_INDEX)
    ok = actual == expected
    passed += ok
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {prefix:<18} AS{origin:<6} got {actual:<9} "
          f"expected {expected:<9} ({description})")

print(f"\n{passed}/{len(INDEX_TEST_CASES)} cases passed")

### Task 3.3 — Classify the whole routing table

> **Gotcha:** `annotate_rov()` caches the *output of your classifier* under `data/`. If you
> change `classify_rov_indexed` after this cell has run, rerun it with `force_refresh=True`;
> otherwise Tasks 4–6 quietly analyze your old states.

In [ ]:
def annotate_rov(bgp_df: pd.DataFrame, vrp_index, date: str = SNAPSHOT_DATE,
                 force_refresh: bool = False) -> pd.DataFrame:
    """Return a copy of bgp_df with a rov_state column, cached on disk."""
    cache_path = DATA_DIR / f"rov-annotated-{date}.csv.gz"
    if cache_path.exists() and not force_refresh:
        return pd.read_csv(cache_path)

    states = []
    for count, (prefix, origin) in enumerate(zip(bgp_df["prefix"], bgp_df["origin"]), 1):
        states.append(classify_rov_indexed(prefix, int(origin), vrp_index))
        if count % 250_000 == 0:
            print(f"  classified {count:>9,} / {len(bgp_df):,}")

    annotated = bgp_df.copy()
    annotated["rov_state"] = states
    annotated.to_csv(cache_path, index=False, compression="gzip")
    return annotated


bgp_classified = annotate_rov(bgp, vrp_index)
bgp_classified.head()

---
## Task 4: How Much of the Internet Implements RPKI?

Detailed guidance: [Task 4 — ROV Shares](Task-4-rov-shares.md)

Across everything announced on `SNAPSHOT_DATE`, how much does RPKI vouch for?

### Task 4.1 — Internet-wide ROV state shares

In [ ]:
# YOUR CODE HERE
# Produce: rov_summary — a DataFrame indexed by ROV state ("Valid", "Invalid", "Not Found"),
#          with columns count (number of announcements) and percent (share of all, 0-100).
#          Then draw a bar chart of rov_summary["percent"] with a labeled y-axis and a title.
# Hint: bgp_classified["rov_state"].value_counts(), and the same call with normalize=True.
# Hint: .reindex(["Valid", "Invalid", "Not Found"]) fixes the row order.

**Q9** Report your headline numbers: what share of announced (prefix, origin) pairs is **Valid**, **Not Found**, and **Invalid**? Operators who enforce ROV drop Invalid routes — so who is actually affected by the Invalid share you measured, and why is that share so much smaller than the other two? What does the Not Found share say about how much of the internet would go dark if operators dropped Not Found routes today?

*Your answer for Q9:*

Replace this line with your answer.

---
## Task 5: ROA Coverage, One Network at a Time

Detailed guidance: [Task 5 — Per-AS Coverage](Task-5-per-as-coverage.md)

Deployment is a **per-operator decision**, so the more telling unit is the origin AS: what fraction of *its own* announcements does RPKI vouch for?

### Task 5.1 — Per-AS ROA coverage

In [ ]:
# YOUR CODE HERE
# Produce: per_as — a DataFrame indexed by origin ASN with columns
#          n_announcements (int: pairs the AS originates) and valid_frac (float in [0, 1]:
#          the fraction of them classified "Valid").
#          Then draw a histogram of per_as["valid_frac"] (20 bins) with labeled axes.
# Hint: the mean of a boolean column is the fraction of True values.
# Hint: groupby("origin").agg(...) with named aggregations keeps the ASN as the index.

**Q10** Describe the shape of the histogram. Most ASes sit at one of the two extremes — all or nothing — rather than in the middle. Why does ROA publishing, as an operational decision, produce exactly this shape? What kind of AS would you expect to find *in* the middle?

*Your answer for Q10:*

Replace this line with your answer.

---
## Task 6: MANRS Participants Versus Everyone Else

Detailed guidance: [Task 6 — MANRS](Task-6-manrs.md)

Every ASN in `manrs_asns` belongs to an operator that **publicly committed** to publishing ROAs. If that commitment means anything, it should be visible in data the participants don't control.

### Task 6.1 — Announcement-level comparison

In [ ]:
# YOUR CODE HERE
# Produce: a boolean column bgp_classified["manrs"] (origin is in manrs_asns), and
#          manrs_crosstab — a DataFrame with one row per group (False, True) and one column
#          per ROV state, each row giving percentages that sum to 100.
# Hint: Series.isin(manrs_asns) builds the label.
# Hint: pd.crosstab(..., normalize="index") normalizes within each row.

### Task 6.2 — Per-AS summary

In [ ]:
# YOUR CODE HERE
# Produce: a boolean column per_as["manrs"], and manrs_summary — a DataFrame indexed by
#          manrs (False, True) with columns n_ases, median_valid_frac, share_zero
#          (share of the group's ASes with valid_frac == 0), and share_full (== 1).
# Hint: per_as.index.isin(manrs_asns) labels the ASes.
# Hint: groupby("manrs").agg(...) with named aggregations; a lambda can compute a share.

### Task 6.3 — Comparing the distributions

In [ ]:
# YOUR CODE HERE
# Produce: one figure with an empirical CDF of per_as["valid_frac"] for each group
#          (MANRS and non-MANRS), with a legend and labeled axes.
# Hint: for each group, sort its valid_frac values and plot them against
#       np.linspace(0, 1, len(values)).

**Q11** Quantify the gap: how do the two groups compare on `median_valid_frac` and `share_zero`? This is an observational comparison, not an experiment. Give one mechanism by which MANRS participation *causes* better ROA coverage, and one by which the causality runs the other way (self-selection). What data would help you tell the two apart?

*Your answer for Q11:*

Replace this line with your answer.

---
## Task 7: The BGPsec and ASPA Footprint

Detailed guidance: [Task 7 — BGPsec Footprint](Task-7-bgpsec-footprint.md)

ROAs measure origin validation's *publish* side. The BGPsec router certificate and the ASPA record are the corresponding first deployable artifacts of path security; the background is in [Introduction.md](Introduction.md).

### Task 7.1 — Count the path-security artifacts

In [ ]:
router_asns = sorted({key["asn"] for key in snapshot["bgpsec_keys"]})

print(f"Validated RPKI as of {snapshot['buildtime']}\n")
print(f"  ROAs                        {snapshot['roas']:>9,}")
print(f"  VRPs                        {snapshot['vrps']:>9,}")
print(f"  ASPA records                {snapshot['aspas']:>9,}")
print(f"  BGPsec router certificates  {len(snapshot['bgpsec_keys']):>9,}")
print(f"\nASNs with at least one BGPsec router certificate: {router_asns}")

**Q12** Put the two counts side by side: ROAs versus BGPsec router certificates. Explain the gap in terms of what a network *gets* for publishing each one today — one is a unilateral act with immediate benefit, the other buys nothing until neighbors join. Compare the ASPA count as well: ASPA is years younger than BGPsec. What does its count suggest about which kinds of path-security design can actually deploy?

*Your answer for Q12:*

Replace this line with your answer.

### Task 7.2 — Reflect

**Q13** Your MANRS list is from today; the routing table and VRPs are from `SNAPSHOT_DATE`. Suppose a network published its first ROAs in the gap between the two dates, or joined MANRS last week. Which direction would each case push your MANRS comparison — overstate or understate the participants' lead?

*Your answer for Q13:*

Replace this line with your answer.

**Q14** This notebook measured *publishing* — MANRS Action 4. The other half of origin validation, networks actually *enforcing* ROV on the routes they hear (Action 1), is invisible in every dataset you used. Why? Sketch how you would measure ROV enforcement from the outside: what would you need to announce, and what would you watch for?

*Your answer for Q14:*

Replace this line with your answer.